# PubliBike-Fahrten: Analyse und Karte

Dieses Notebook arbeitet auf `data/rides_geo.csv` — der Datei, die
`python3 -m publibike_rides` erzeugt. Fehlt sie, greift es auf den
synthetischen Beispieldatensatz unter `data/beispiel/` zurueck, damit es auch
ohne PubliBike-Konto durchlaeuft. Welche Quelle benutzt wurde, steht in der
Ausgabe der ersten Zelle.

**Zur Distanz:** Die Spalten `distance_km_luftlinie` und `speed_kmh_luftlinie`
messen die gerade Linie zwischen Start- und Zielstation. Die tatsächlich
gefahrene Strecke ist länger; die API liefert keine GPS-Spur. Alles, was auf
diesen Spalten aufbaut, ist also eine Untergrenze, keine Fahrleistung.

In [ ]:
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import pandas as pd
from folium.plugins import HeatMap

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "data"

# Eigene Fahrten, wenn vorhanden — sonst der synthetische Beispieldatensatz.
# So laeuft das Notebook auch bei jemandem ohne PubliBike-Konto durch.
eigene = DATA / "rides_geo.csv"
beispiel = DATA / "beispiel" / "rides_geo.csv"
quelle = eigene if eigene.exists() else beispiel

if not quelle.exists():
    raise SystemExit(
        "Keine Daten gefunden. Eigene holen mit: python3 -m publibike_rides  "
        "- oder den Beispieldatensatz erzeugen mit: python3 scripts/beispieldaten.py"
    )

rides = pd.read_csv(quelle, parse_dates=["start_date", "end_date"])
ist_beispiel = quelle == beispiel
print("Quelle:", quelle.relative_to(REPO), "(erfundene Daten)" if ist_beispiel else "(eigene Fahrten)")
print(f"{len(rides)} Mieten von {rides['date'].min()} bis {rides['date'].max()}")
rides.head(3)

## Überblick

In [ ]:
# Durchgehend die exakten Sekunden, nicht das aufgerundete Minutenfeld.
rides["dauer_min"] = rides["duration_seconds"] / 60
dauer = rides["dauer_min"]
uebersicht = pd.Series({
    "Mieten": len(rides),
    "Wege": rides["journey_id"].nunique(),
    "Zeitraum (Tage)": (rides["start_date"].max() - rides["start_date"].min()).days,
    "Aktive Tage": rides["date"].nunique(),
    "Stunden gesamt": round(dauer.sum() / 60, 1),
    "Dauer Median (min)": round(dauer.median(), 1),
    "Dauer Mittel (min)": round(dauer.mean(), 1),
    "Laengste Fahrt (min)": round(dauer.max(), 1),
    "Luftlinie gesamt (km)": round(rides["distance_km_luftlinie"].sum(), 1),
    "Genutzte Stationen": pd.concat([rides["start_station"], rides["end_station"]]).nunique(),
    "Rundfahrten": int(rides["is_round_trip"].sum()),
})
uebersicht.to_frame("Wert")

## Datenqualität zuerst

Bevor gerechnet wird: Was ist auffällig? Eine Fahrt zerfällt oft in zwei
Mieten — beginnt die nächste innerhalb von drei Minuten, war es ein Weg.
Dafür gibt es drei Gründe, die sich in den Daten unterscheiden lassen.

| Markierung | Was passiert ist | Erkennungsmerkmal |
|---|---|---|
| `defekt_sofort` | Das Velo lief gar nicht erst. | Miete endet an ihrer Startstation. |
| `defekt_unterwegs` | Das Velo fiel nach kurzer Fahrt aus. | Ging woanders hin, höchstens 20 Minuten. |
| `gratiszeit_zurueckgesetzt` | Kein Defekt — neue Miete vor Ablauf der freien 30 Minuten. | Länger als 20 Minuten. |
| `tempo_unplausibel` | Buchungsartefakt, keine gefahrene Strecke. | Über 40 km/h Luftlinie. |
| `station_unbekannt` | Keine Koordinaten im GBFS-Feed. | Distanz bleibt leer statt geraten. |

Mehrere Markierungen pro Zeile sind möglich, getrennt durch Semikolon.
Gerechnet wird durchgehend mit `duration_seconds` aus den Zeitstempeln, nicht
mit `duration_min` — das Feld der API ist auf ganze Minuten **aufgerundet**.

In [ ]:
auffaellig = rides[rides["quality_flag"].notna() & (rides["quality_flag"] != "")]
print(f"{len(auffaellig)} von {len(rides)} Mieten markiert\n")

for marke in ["defekt_sofort", "defekt_unterwegs", "gratiszeit_zurueckgesetzt",
              "tempo_unplausibel", "station_unbekannt"]:
    treffer = rides["quality_flag"].fillna("").str.split(";").apply(lambda f: marke in f)
    print(f"  {marke:<28} {treffer.sum():>3}")

### Wie gut trennen die Schwellen?

Beide Grenzen der Regel lassen sich an den Daten prüfen. Wenn sie taugen,
liegen sie in einer Lücke der Verteilung und nicht mitten im Gedränge —
dann ändert ein anderer Wert am Ergebnis nichts.

In [ ]:
kette = rides[rides["gap_to_next_min"] <= 3].copy()
kette["minuten"] = kette["duration_seconds"] / 60

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))

# Links: die Luecke zur naechsten Miete
axes[0].scatter(rides["gap_to_next_min"].clip(upper=10), [0] * len(rides),
                alpha=0.4, s=40, color="#6E2C50")
axes[0].axvline(3, color="#C0392B", linestyle="--", label="Schwelle 3 Min")
axes[0].set_xlabel("Luecke zur naechsten Miete (Minuten, bei 10 abgeschnitten)")
axes[0].set_yticks([])
axes[0].legend()
axes[0].set_title("Verkettung: wo liegt die Grenze?")

# Rechts: die Fahrzeit der verketteten Mieten
for name, teil in kette.groupby(kette["is_round_trip"].map(
        {True: "kam nicht vom Fleck", False: "fuhr woanders hin"})):
    axes[1].scatter(teil["minuten"], [name] * len(teil), alpha=0.6, s=50, color="#6E2C50")
axes[1].axvline(20, color="#C0392B", linestyle="--", label="Schwelle 20 Min")
axes[1].set_xlabel("Fahrzeit der Miete (Minuten)")
axes[1].legend()
axes[1].set_title("Defekt oder Gratiszeit?")

plt.tight_layout()
plt.show()

print("Fahrzeiten der verketteten Mieten (Minuten), sortiert:")
print(sorted(kette["minuten"].round(1)))

## Mieten oder Wege?

Zusammenhängende Mieten tragen dieselbe `journey_id`. Das ändert die
Kennzahlen: «Wie oft bin ich gefahren» beantwortet die Zahl der Wege, nicht
die der Mieten.

In [ ]:
wege = rides.groupby("journey_id").agg(
    mieten=("ride_id", "count"),
    start=("start_date", "min"),
    dauer_min=("duration_seconds", lambda s: round(s.sum() / 60, 1)),
    distanz_km=("distance_km_luftlinie", "sum"),
)

print(f"{len(rides)} Mieten -> {len(wege)} Wege")
print()
print("Mieten pro Weg:")
print(wege["mieten"].value_counts().sort_index().to_string())
print()
print("Wege mit mehr als einer Miete, laengste zuerst:")
display(wege[wege["mieten"] > 1].sort_values("mieten", ascending=False).head(8))

In [ ]:
# Fuer Tempo- und Streckenauswertungen die Artefakte rauswerfen.
flags = rides["quality_flag"].fillna("")
sauber = rides[~flags.str.contains("tempo_unplausibel")]
print(f"{len(rides)} Mieten gesamt")
print(f"{len(sauber)} ohne Buchungsartefakte")
print(f"{len(wege)} zusammenhaengende Wege")
echte_fahrten = sauber

## Wann wird gefahren?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

pro_monat = rides.groupby("month").size()
pro_monat.plot(kind="bar", ax=axes[0], color="#6E2C50")
axes[0].set_title("Fahrten pro Monat")
axes[0].set_xlabel("")
axes[0].set_ylabel("Fahrten")

reihenfolge = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"]
pro_wochentag = rides["weekday"].value_counts().reindex(reihenfolge, fill_value=0)
pro_wochentag.plot(kind="bar", ax=axes[1], color="#6E2C50")
axes[1].set_title("Fahrten pro Wochentag")
axes[1].set_xlabel("")

pro_stunde = rides["hour"].value_counts().sort_index().reindex(range(24), fill_value=0)
pro_stunde.plot(kind="bar", ax=axes[2], color="#6E2C50")
axes[2].set_title("Fahrten pro Tagesstunde")
axes[2].set_xlabel("Stunde")

plt.tight_layout()
plt.show()

## Dauer und Velotyp

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

echte_fahrten["dauer_min"].plot(kind="hist", bins=30, ax=axes[0], color="#6E2C50")
axes[0].set_title("Verteilung der Fahrtdauer")
axes[0].set_xlabel("Minuten")

rides["vehicle_label"].value_counts().plot(kind="barh", ax=axes[1], color="#6E2C50")
axes[1].set_title("E-Bike oder Velo")
axes[1].set_xlabel("Fahrten")

plt.tight_layout()
plt.show()

print("Tempo Luftlinie (ohne markierte Fahrten, km/h):")
print(sauber["speed_kmh_luftlinie"].describe().round(1).to_string())

## Die wichtigsten Stationen

In [ ]:
starts = rides["start_station"].value_counts().rename("Starts")
ziele = rides["end_station"].value_counts().rename("Ziele")
stationen = pd.concat([starts, ziele], axis=1).fillna(0).astype(int)
stationen["Gesamt"] = stationen["Starts"] + stationen["Ziele"]
stationen.sort_values("Gesamt", ascending=False).head(12)

In [ ]:
strecken = (
    rides.groupby(["start_station", "end_station"])
    .agg(fahrten=("ride_id", "count"), dauer_median=("duration_min", "median"))
    .sort_values("fahrten", ascending=False)
)
strecken.head(12)

## Karte

Zwei Ebenen: Kreise für die Stationen, Grösse nach Nutzung, und Linien für die
häufigsten Verbindungen. Die Linien sind gerade — sie zeigen, welche Stationen
verbunden sind, nicht den gefahrenen Weg.

In [ ]:
koordinaten = {}
for _, zeile in rides.iterrows():
    if pd.notna(zeile["start_lat"]):
        koordinaten[zeile["start_station"]] = (zeile["start_lat"], zeile["start_lon"])
    if pd.notna(zeile["end_lat"]):
        koordinaten[zeile["end_station"]] = (zeile["end_lat"], zeile["end_lon"])

nutzung = stationen["Gesamt"].to_dict()
mitte = (
    sum(p[0] for p in koordinaten.values()) / len(koordinaten),
    sum(p[1] for p in koordinaten.values()) / len(koordinaten),
)

karte = folium.Map(location=mitte, zoom_start=13, tiles="OpenStreetMap")

# Verbindungen: je dicker, desto haeufiger gefahren
top_strecken = strecken[strecken.index.get_level_values(0) != strecken.index.get_level_values(1)].head(25)
maximum = top_strecken["fahrten"].max()
for (start, ziel), zeile in top_strecken.iterrows():
    if start in koordinaten and ziel in koordinaten:
        folium.PolyLine(
            [koordinaten[start], koordinaten[ziel]],
            weight=1 + 6 * zeile["fahrten"] / maximum,
            color="#6E2C50",
            opacity=0.45,
            tooltip=f"{start} → {ziel}: {int(zeile['fahrten'])} Fahrten",
        ).add_to(karte)

# Stationen
for name, (lat, lon) in koordinaten.items():
    anzahl = nutzung.get(name, 0)
    folium.CircleMarker(
        location=(lat, lon),
        radius=3 + 12 * anzahl / max(nutzung.values()),
        color="#6E2C50",
        fill=True,
        fill_color="#C39BD3",
        fill_opacity=0.8,
        weight=1,
        tooltip=f"{name}: {anzahl} mal genutzt",
    ).add_to(karte)

karte

### Wärmekarte

Wo verdichtet sich die Nutzung? Jede Station geht mit ihrer Fahrtenzahl als
Gewicht ein.

In [ ]:
heat = folium.Map(location=mitte, zoom_start=13, tiles="OpenStreetMap")
HeatMap(
    [[lat, lon, nutzung.get(name, 0)] for name, (lat, lon) in koordinaten.items()],
    radius=25,
    blur=18,
).add_to(heat)
heat

## Karten speichern

Die Dateien landen in `maps/` und bleiben über `.gitignore` aus dem Repository
draussen — sie enthalten dieselben persönlichen Bewegungsdaten wie das CSV.

In [ ]:
ziel = REPO / "maps"
ziel.mkdir(exist_ok=True)
karte.save(str(ziel / "stationen.html"))
heat.save(str(ziel / "waermekarte.html"))
print(f"Gespeichert in {ziel}")